In [ ]:
import torch # pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2, os, json, gc, math
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from utils.Plotter.index import Plotter
from Network.index import ModelNetwork
from utils.index import *

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
print(torch.cuda.get_device_name(0))  # nome da GPU
pd.set_option('display.max_columns', None)

# HISTÓRICO DE TREINAMENTO

In [ ]:
df = []

for bakup in os.listdir('Backup/'):
    path = f'Backup/{bakup}/info.json'

    with open(path, 'r', encoding='utf-8') as file:
        file = json.loads(file.read())

    info = {}
    for key, value in file.items():
        info.update(value)

    if not info.get('dataset'):
        info['dataset'] = 'facies'

    info['id'] = int(info.get('path').split('_')[-1])
    df.append(info)


df = pd.DataFrame(df).sort_values(by='id')
df.head()

### Filtrando o Alvo

In [ ]:
df['total_images'] = [n_images.get('total') for n_images in df.n_images.values]
df = df.loc[df.dataset == 'facies']
df.head(2)

# ANÁLISE DE VARIAÇÕES

In [ ]:
class VariationAnalysis:
    def __init__(self, df, variations, id=None, metric='test_iou'):
        self.df = df.copy()
        self.variations = [v for v in variations if v in self.df.columns]
        self.id = id
        self.metric = metric
        str_cols    = self.df.select_dtypes(include=['object']).columns
        self.df[str_cols] = self.df[str_cols].astype(str)

    def getSorted(self, df_group, variation):
        df_agg = df_group.groupby(variation, as_index=False)[self.metric].max()

        numeric_col = pd.to_numeric(df_agg[variation], errors='coerce')
        if numeric_col.notna().all():
            df_agg['__sort'] = numeric_col
            df_agg = df_agg.sort_values('__sort').drop(columns=['__sort'])
        else:
            df_agg = df_agg.sort_values(variation)

        return dict(zip(df_agg[variation], df_agg[self.metric]))

    def plot(self):
        for var in self.variations:
            fixed_cols = [c for c in self.variations if c != var]
            grouped    = self.df.groupby(fixed_cols) if fixed_cols else [(None, self.df)]

            for _, group in grouped:
                if group[var].nunique() <= 1:
                    continue

                fixed_dict = group.iloc[0][fixed_cols].to_dict() if fixed_cols else {}
                print(f"target: {var} | fixed: {fixed_dict}")

                ref      = [self.id] if self.id else []
                df_table = group[self.variations + ref + [self.metric]].copy()
                df_table[self.metric] = df_table[self.metric].round(4)

                plt.figure(figsize=(20, 5))
                plt.subplot(1, 2, 1)
                table = plt.table(cellText=df_table.astype(str).values, colLabels=df_table.columns, loc='center', cellLoc='center')
                table.auto_set_font_size(False)
                table.set_fontsize(8); table.scale(1, 1.5)
                plt.title(f'Variation in: {var}')
                plt.axis('off')

                plt.subplot(1, 2, 2)
                Plotter(self.getSorted(group, var), title=f'Comparação - Variação em {var}')
                plt.ylabel('Mean IoU')
                plt.show()


variations = ['loss', 'network', 'lr', 'img_size', 'tile_size', 'n_aug', 'scheduler', 'num_filters', 'dropout', 'radius'] # 'total_images']
analysis   = VariationAnalysis(df, variations, id='path', metric='test_iou')
analysis.plot()